In [1]:
# ============================================================
#  PIPELINE COMPLETO — procesa un vídeo de principio a fin
#  Para cambiar de vídeo: edita SOLO la variable VIDEO_FILE
# ============================================================
import sys, json, cv2
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
import torch

REPO_ROOT = Path("../..").resolve()

# ---- ÚNICO parámetro a cambiar por vídeo ----
VIDEO_FILE = "test (27).mp4"          # <--- cambia esto para otro vídeo
# ---------------------------------------------

VIDEO_IN = REPO_ROOT / "data" / "raw" / VIDEO_FILE
NOMBRE = VIDEO_IN.stem.replace(" ", "_").replace("(", "").replace(")", "")

OUT_NPZ = REPO_ROOT / "data" / "tracking" / f"homografias_pnl_{NOMBRE}.npz"
OUT_JSON = REPO_ROOT / "data" / "tracking" / f"integrado_{NOMBRE}.json"
OUT_NPZ.parent.mkdir(parents=True, exist_ok=True)

# Modelos y dependencias
MODELO_DETECTOR = REPO_ROOT / "models" / "best_v4.pt"
PNL_DIR = REPO_ROOT / "external" / "PnLCalib"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Verificaciones de existencia
assert VIDEO_IN.exists(), f"No existe el vídeo: {VIDEO_IN}"
assert MODELO_DETECTOR.exists(), f"No existe el detector: {MODELO_DETECTOR}"
assert PNL_DIR.exists(), f"No existe PnLCalib: {PNL_DIR}"

cap = cv2.VideoCapture(str(VIDEO_IN))
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W_IMG = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_IMG = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS)
cap.release()

print(f"Vídeo: {VIDEO_FILE}")
print(f"  {N_FRAMES} frames, {W_IMG}x{H_IMG}, {FPS:.0f} fps")
print(f"  NOMBRE = {NOMBRE}")
print(f"  device = {device}")
print(f"Salidas: {OUT_NPZ.name}, {OUT_JSON.name}")

Vídeo: test (27).mp4
  750 frames, 1920x1080, 25 fps
  NOMBRE = test_27
  device = cuda
Salidas: homografias_pnl_test_27.npz, integrado_test_27.json


In [2]:
# ---- PnLCalib: calibración por puntos+líneas, extraer homografías imagen->campo ----
sys.path.insert(0, str(PNL_DIR))
from model.cls_hrnet import get_cls_net
from model.cls_hrnet_l import get_cls_net as get_cls_net_l
from utils.utils_calib import FramebyFrameCalib
import torchvision.transforms as T
import yaml
import inference as pnl_inference
from inference import inference, projection_from_cam_params

# Cargar modelos de PnLCalib
cfg_kp = yaml.safe_load(open(PNL_DIR / "config" / "hrnetv2_w48.yaml"))
cfg_l  = yaml.safe_load(open(PNL_DIR / "config" / "hrnetv2_w48_l.yaml"))
model = get_cls_net(cfg_kp); model.load_state_dict(torch.load(PNL_DIR/"weights"/"SV_kp", map_location=device)); model.to(device).eval()
model_l = get_cls_net_l(cfg_l); model_l.load_state_dict(torch.load(PNL_DIR/"weights"/"SV_lines", map_location=device)); model_l.to(device).eval()

# Inyectar variables globales que inference() necesita
pnl_inference.device = device
pnl_inference.transform2 = T.Resize((540, 960))
pnl_inference.transform = T.Resize((540, 960))

KP_THRESHOLD, LINE_THRESHOLD, PNL_REFINE = 0.1486, 0.3880, True

def homografia_imagen_a_campo(params):
    """Convierte parámetros de cámara de PnLCalib en homografía píxel->campo (origen ESQUINA)."""
    if params is None or "cam_params" not in params:
        return None
    P = projection_from_cam_params(params)
    M = P[:, [0, 1, 3]]   # columnas para suelo Z=0
    try:
        H_img2centro = np.linalg.inv(M)
    except np.linalg.LinAlgError:
        return None
    T_c2e = np.array([[1,0,52.5],[0,1,34.0],[0,0,1.0]])
    H = T_c2e @ H_img2centro
    return H / H[2,2]

# Recorrer el vídeo
from tqdm import tqdm
cam = FramebyFrameCalib(iwidth=W_IMG, iheight=H_IMG, denormalize=True)
arr = np.full((N_FRAMES, 3, 3), np.nan)
n_ok = 0
cap = cv2.VideoCapture(str(VIDEO_IN))
for f_idx in tqdm(range(N_FRAMES)):
    ret, frame = cap.read()
    if not ret: break
    try:
        params = inference(cam, frame, model, model_l, KP_THRESHOLD, LINE_THRESHOLD, PNL_REFINE)
        H = homografia_imagen_a_campo(params)
    except Exception:
        H = None
    if H is not None:
        arr[f_idx] = H; n_ok += 1
cap.release()

np.savez_compressed(OUT_NPZ, homografias=arr, n_frames=N_FRAMES)
print(f"\nPnLCalib: {n_ok}/{N_FRAMES} frames calibrados ({100*n_ok/N_FRAMES:.0f}%)")
print(f"Guardado: {OUT_NPZ.name}")

# Sanity check: magnitud de las posiciones proyectadas
maxmag = 0
for i in range(N_FRAMES):
    if np.isnan(arr[i]).any(): continue
    pts = np.array([[[W_IMG/2, H_IMG*0.7]]], dtype=np.float32)
    p = cv2.perspectiveTransform(pts, arr[i])[0][0]
    maxmag = max(maxmag, abs(p[0]), abs(p[1]))
print(f"Magnitud máxima proyectada (centro-bajo): {maxmag:.0f}m (sano si <120)")

100%|██████████| 750/750 [12:55<00:00,  1.03s/it]


PnLCalib: 750/750 frames calibrados (100%)
Guardado: homografias_pnl_test_27.npz
Magnitud máxima proyectada (centro-bajo): 2794m (sano si <120)


In [4]:
# ---- Filtro de calibraciones malas: marcar NaN los frames con magnitud explosiva ----
# PnLCalib calibra todos los frames, pero unos pocos dan homografías con horizonte numérico
# cerca de la zona proyectada -> posiciones disparadas. Se descartan (NaN) para que el
# pipeline los trate como "sin calibración" en ese frame.

arr = np.load(OUT_NPZ)["homografias"]
N = len(arr)
UMBRAL_MAG = 120.0   # metros: por encima, la calibración de ese frame no es fiable

descartados = []
for i in range(N):
    if np.isnan(arr[i]).any():
        continue
    pts = np.array([[[x,y]] for x in (480,960,1440) for y in (400,700,950)], dtype=np.float32)
    proj = cv2.perspectiveTransform(pts, arr[i]).reshape(-1,2)
    if np.abs(proj).max() > UMBRAL_MAG:
        arr[i] = np.nan
        descartados.append(i)

# Guardar las homografías filtradas (sobrescribe el .npz)
np.savez_compressed(OUT_NPZ, homografias=arr, n_frames=N)

n_validas = sum(1 for i in range(N) if not np.isnan(arr[i]).any())
print(f"Frames con calibración fiable: {n_validas}/{N} = {100*n_validas/N:.0f}%")
print(f"Descartados por magnitud explosiva (>120m): {len(descartados)} frames")
print(f"  índices: {descartados[:30]}")

# Verificar que ya no hay explosivos
maxmag = 0
for i in range(N):
    if np.isnan(arr[i]).any(): continue
    pts = np.array([[[960, 750]]], dtype=np.float32)
    p = cv2.perspectiveTransform(pts, arr[i])[0][0]
    maxmag = max(maxmag, abs(p[0]), abs(p[1]))
print(f"Magnitud máxima tras filtrar: {maxmag:.0f}m (debe ser <120)")

Frames con calibración fiable: 720/750 = 96%
Descartados por magnitud explosiva (>120m): 30 frames
  índices: [2, 3, 5, 6, 12, 15, 26, 29, 30, 32, 36, 38, 39, 40, 44, 45, 46, 53, 55, 57, 58, 275, 276, 278, 281, 287, 288, 289, 364, 581]
Magnitud máxima tras filtrar: 106m (debe ser <120)


In [8]:
# ---- Tracking BoT-SORT + cosido de fragmentos + filtro de espurios ----
from ultralytics import YOLO
from collections import deque

detector = YOLO(str(MODELO_DETECTOR))
print(f"Clases del detector: {detector.names}")

# Config del tracker (se crea si no existe) — TUS parámetros, no el default
tracker_cfg = REPO_ROOT / "configs" / "botsort_custom.yaml"
tracker_cfg.parent.mkdir(parents=True, exist_ok=True)
if not tracker_cfg.exists():
    tracker_cfg.write_text("""tracker_type: botsort
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.6
track_buffer: 120
match_thresh: 0.8
fuse_score: true
gmc_method: sparseOptFlow
proximity_thresh: 0.5
appearance_thresh: 0.25
with_reid: true
model: auto
""", encoding="utf-8")

# Pasada de tracking
frames_dets = []
results = detector.track(source=str(VIDEO_IN), stream=True, persist=True,
                         tracker=str(tracker_cfg), conf=0.25, imgsz=1280, verbose=False)
for n, rd in enumerate(results):
    dets = []
    if rd.boxes is not None and rd.boxes.id is not None:
        for (x1, y1, x2, y2), tid, c in zip(rd.boxes.xyxy.cpu().numpy(),
                                            rd.boxes.id.cpu().numpy().astype(int),
                                            rd.boxes.cls.cpu().numpy().astype(int)):
            dets.append((int(tid), float(x1), float(y1), float(x2), float(y2), int(c)))
    frames_dets.append(dets)
    if (n + 1) % 100 == 0:
        print(f"Tracking: {n + 1} frames...")
N_FRAMES = len(frames_dets)

# Construir tracks completos por id
tracks_full = defaultdict(list)
for f_idx, dets in enumerate(frames_dets):
    for (tid, x1, y1, x2, y2, c) in dets:
        tracks_full[tid].append((f_idx, (x1 + x2) / 2, y2, int(c)))

# Dividir cada track por cambio de clase sostenido (>=MIN_SEG)
MIN_SEG = 15
tracks2, clase2, mapa_det = {}, {}, {}
next_id = 1
for tid, pts in tracks_full.items():
    segs, actual, pendiente, c_actual = [], [], [], None
    for p in pts:
        c = p[3]
        if c_actual is None:
            c_actual = c
        if c == c_actual:
            if pendiente:
                actual.extend([(f, x, y, c_actual) for (f, x, y, _) in pendiente]); pendiente = []
            actual.append(p)
        else:
            pendiente.append(p)
            if len(pendiente) >= MIN_SEG:
                if actual:
                    segs.append((c_actual, actual))
                c_actual, actual, pendiente = c, [(f, x, y, c) for (f, x, y, _) in pendiente], []
    if pendiente:
        actual.extend([(f, x, y, c_actual) for (f, x, y, _) in pendiente])
    if actual:
        segs.append((c_actual, actual))
    for c_seg, seg in segs:
        nid = next_id; next_id += 1
        tracks2[nid] = [(f, x, y) for (f, x, y, c) in seg]
        clase2[nid] = c_seg
        for (f, x, y, c) in seg:
            mapa_det[(f, tid)] = nid

# Cosido: fusionar segmentos consecutivos del mismo objeto por proximidad/predicción
MAX_GAP, MAX_SOLAPE, RADIO_BASE, RADIO_POR_FRAME, RADIO_SOLAPE = 75, 12, 50, 5, 35
resumen, pos_pf = {}, {}
for nid, pts in tracks2.items():
    if clase2[nid] not in (1, 2) or len(pts) < 3:
        continue
    arr = np.array(pts); ult = arr[-5:]
    dt = ult[-1, 0] - ult[0, 0]
    vel = (ult[-1, 1:] - ult[0, 1:]) / dt if dt > 0 else np.zeros(2)
    resumen[nid] = {"ini": int(arr[0, 0]), "fin": int(arr[-1, 0]),
                    "pos_ini": arr[0, 1:], "pos_fin": arr[-1, 1:], "vel": vel}
    pos_pf[nid] = {int(f): np.array([x, y]) for f, x, y in pts}

cands = []
for ta, ra in resumen.items():
    for tb, rb in resumen.items():
        if ta == tb or clase2[ta] != clase2[tb]:
            continue
        gap = rb["ini"] - ra["fin"]
        if gap > MAX_GAP or gap < -MAX_SOLAPE:
            continue
        if gap > 0:
            d = np.linalg.norm(rb["pos_ini"] - (ra["pos_fin"] + ra["vel"] * gap))
            r = RADIO_BASE + RADIO_POR_FRAME * gap
        else:
            pa = pos_pf[ta].get(rb["ini"])
            if pa is None:
                continue
            d, r = np.linalg.norm(rb["pos_ini"] - pa), RADIO_SOLAPE
        if d <= r:
            cands.append((d, ta, tb))
cands.sort()
uf, ui, fusion = set(), set(), {}
for d, ta, tb in cands:
    if ta in uf or tb in ui:
        continue
    fusion[tb] = ta; uf.add(ta); ui.add(tb)
def raiz(t):
    while t in fusion:
        t = fusion[t]
    return t
mapa_final = {nid: raiz(nid) for nid in resumen}

# Filtro de segmentos espurios (cortos que solapan con otro de su clase)
MIN_DET_SEG = 5
rango_seg = {nid: (int(np.array(pts)[0, 0]), int(np.array(pts)[-1, 0]))
             for nid, pts in tracks2.items() if len(pts) > 0}
raiz_de = {nid: int(mapa_final.get(nid, nid)) for nid in tracks2}
espurios = set()
for nid, pts in tracks2.items():
    if len(pts) >= MIN_DET_SEG or nid not in rango_seg:
        continue
    a0, a1 = rango_seg[nid]; cls_a = clase2[nid]
    for otro, pts2 in tracks2.items():
        if otro == nid or clase2[otro] != cls_a or len(pts2) < MIN_DET_SEG:
            continue
        if raiz_de[otro] == raiz_de[nid]:
            continue
        b0, b1 = rango_seg[otro]
        if not (a1 < b0 or a0 > b1):
            espurios.add(nid); break

# Construir frames_track final con los ids cosidos
frames_track = []
for f_idx, dets in enumerate(frames_dets):
    fr = []
    for (tid, x1, y1, x2, y2, c) in dets:
        if c in (1, 2):
            nid = mapa_det.get((f_idx, tid), tid)
            if nid in espurios:
                continue
            idf = int(mapa_final.get(nid, nid))
        else:
            idf = int(tid)
        fr.append({"id": idf, "bbox": [x1, y1, x2, y2], "cls": c})
    frames_track.append(fr)

n_ids = len({d['id'] for fr in frames_track for d in fr if d['cls'] in (1, 2)})
cls_cnt = Counter(d["cls"] for fr in frames_track for d in fr)
print(f"\nTracking + cosido: {n_ids} IDs de persona (vs 157 sin cosido)")
print(f"  detecciones por clase: {dict(cls_cnt)}")
print(f"  espurios filtrados: {len(espurios)}")

Clases del detector: {0: 'ball', 1: 'player', 2: 'referee'}
Tracking: 100 frames...
Tracking: 200 frames...
Tracking: 300 frames...
Tracking: 400 frames...
Tracking: 500 frames...
Tracking: 600 frames...
Tracking: 700 frames...

Tracking + cosido: 49 IDs de persona (vs 157 sin cosido)
  detecciones por clase: {1: 16313, 2: 279, 0: 134}
  espurios filtrados: 14


In [9]:
# ---- Detección dedicada del BALÓN (conf baja + filtro de trayectoria) ----
# El balón se detecta con conf ~0.5 pero el tracker lo descarta (new_track_thresh=0.6).
# Se detecta directamente con conf 0.3 y se enlaza por proximidad.

CONF_BALON = 0.30
MAX_SALTO_BALON_PX = 250

detec_balon = {}
cap = cv2.VideoCapture(str(VIDEO_IN))
f_idx = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    res = detector(frame, conf=CONF_BALON, imgsz=1280, verbose=False, classes=[0])[0]
    if res.boxes is not None and len(res.boxes) > 0:
        confs = res.boxes.conf.cpu().numpy(); boxes = res.boxes.xyxy.cpu().numpy()
        k = int(np.argmax(confs))
        x1,y1,x2,y2 = boxes[k]
        detec_balon[f_idx] = ((x1+x2)/2, y2, float(confs[k]))
    f_idx += 1
cap.release()
print(f"Balón detectado (conf>={CONF_BALON}): {len(detec_balon)}/{N_FRAMES} = {100*len(detec_balon)/N_FRAMES:.0f}%")

# Filtrar falsos por trayectoria
fs = sorted(detec_balon); a_descartar = set()
for i in range(1, len(fs)):
    fa, fb = fs[i-1], fs[i]
    if fa in a_descartar:
        j = i-1
        while j >= 0 and fs[j] in a_descartar: j -= 1
        if j < 0: continue
        fa = fs[j]
    if fb - fa > 3: continue
    xa,ya,ca = detec_balon[fa]; xb,yb,cb = detec_balon[fb]
    if np.hypot(xb-xa, yb-ya)/(fb-fa) > MAX_SALTO_BALON_PX:
        a_descartar.add(fa if ca < cb else fb)
for f in a_descartar: detec_balon.pop(f, None)
print(f"Tras filtro de trayectoria: {len(detec_balon)}/{N_FRAMES} = {100*len(detec_balon)/N_FRAMES:.0f}%")

# Inyectar el balón en frames_track (id fijo -100)
ID_BALON = -100
for f_idx in range(N_FRAMES):
    frames_track[f_idx] = [d for d in frames_track[f_idx] if d["cls"] != 0]
    if f_idx in detec_balon:
        xc, yp, _ = detec_balon[f_idx]
        frames_track[f_idx].append({"id": ID_BALON, "bbox": [xc-8, yp-16, xc+8, yp], "cls": 0})
print(f"Balones inyectados: {sum(1 for fr in frames_track for d in fr if d['cls']==0)}")

Balón detectado (conf>=0.3): 549/750 = 73%
Tras filtro de trayectoria: 504/750 = 67%
Balones inyectados: 504


In [13]:
import numpy as np, cv2
from collections import defaultdict
arr = np.load(OUT_NPZ)["homografias"]

pos_campo = defaultdict(list)
for fi, fr in enumerate(frames_track):
    H = arr[fi]
    if np.isnan(H).any(): continue
    for d in fr:
        if d["cls"]==1:
            x1,y1,x2,y2 = d["bbox"]
            pie = np.array([[[(x1+x2)/2, y2]]], dtype=np.float32)
            X,Y = cv2.perspectiveTransform(pie, H)[0][0]
            pos_campo[d["id"]].append((X,Y))

vida = defaultdict(int)
for fr in frames_track:
    for d in fr:
        if d["cls"]==1: vida[d["id"]] += 1
titulares = sorted([i for i in vida if vida[i]>=200], key=lambda i:-vida[i])

print("Posicion media de cada titular (X,Y en metros):")
dentro = fuera = 0
for tid in titulares:
    if not pos_campo[tid]: continue
    p = np.median(pos_campo[tid], axis=0)
    en = -3<=p[0]<=108 and -3<=p[1]<=71
    estado = "DENTRO" if en else "FUERA/BORDE"
    if en: dentro += 1
    else: fuera += 1
    print(f"  ID {tid} ({vida[tid]} det): ({p[0]:5.0f},{p[1]:5.0f}) {estado}")
print(f"\nDentro del campo: {dentro} | Fuera/borde: {fuera}")

Posicion media de cada titular (X,Y en metros):
  ID 3 (750 det): (   43,   34) DENTRO
  ID 9 (750 det): (   63,   35) DENTRO
  ID 11 (750 det): (   69,   25) DENTRO
  ID 1 (746 det): (   43,   60) DENTRO
  ID 7 (744 det): (   60,   50) DENTRO
  ID 17 (744 det): (   47,   28) DENTRO
  ID 18 (744 det): (   42,   52) DENTRO
  ID 8 (731 det): (   47,   57) DENTRO
  ID 20 (730 det): (   62,   53) DENTRO
  ID 22 (728 det): (   55,   50) DENTRO
  ID 19 (725 det): (   52,   36) DENTRO
  ID 6 (722 det): (   62,   41) DENTRO
  ID 21 (721 det): (   48,   47) DENTRO
  ID 5 (711 det): (   71,   54) DENTRO
  ID 16 (688 det): (   42,   43) DENTRO
  ID 28 (677 det): (   41,   44) DENTRO
  ID 32 (632 det): (   72,   39) DENTRO
  ID 15 (555 det): (   50,   37) DENTRO
  ID 13 (543 det): (   48,   21) DENTRO
  ID 56 (412 det): (   47,   26) DENTRO
  ID 12 (408 det): (   66,   19) DENTRO
  ID 4 (392 det): (   65,   62) DENTRO
  ID 54 (372 det): (   51,   64) DENTRO
  ID 14 (369 det): (   94,   33) DENTRO


In [14]:
# ---- Proyección de detecciones a coordenadas de campo con PnLCalib ----
arr = np.load(OUT_NPZ)["homografias"]

campo_det = {}   # (f_idx, id) -> (X, Y) en metros
for f_idx, fr in enumerate(frames_track):
    H = arr[f_idx]
    if np.isnan(H).any():
        continue
    for d in fr:
        x1, y1, x2, y2 = d["bbox"]
        pie = np.array([[[(x1 + x2) / 2, y2]]], dtype=np.float32)
        X, Y = cv2.perspectiveTransform(pie, H)[0][0]
        campo_det[(f_idx, d["id"])] = (float(X), float(Y))

print(f"Detecciones proyectadas a campo: {len(campo_det)}")
print(f"Frames sin homografía (saltados): {sum(1 for i in range(N_FRAMES) if np.isnan(arr[i]).any())}")

Detecciones proyectadas a campo: 16393
Frames sin homografía (saltados): 30


In [15]:
# ---- Clasificación de equipos: BGR por identidad + K-means ----
def color_torso(frame, bbox):
    x1, y1, x2, y2 = [int(v) for v in bbox]
    hb, wb = y2 - y1, x2 - x1
    if hb < 10 or wb < 4: return None
    crop = frame[max(y1+int(0.22*hb),0):y1+int(0.50*hb), max(x1+int(0.25*wb),0):x2-int(0.25*wb)]
    if crop.size == 0: return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    pix = crop[cv2.inRange(hsv, (35,40,40), (85,255,255)) == 0]
    if len(pix) < 30: return None
    return np.median(pix.astype(float), axis=0)

col_por_id = defaultdict(list)
cap = cv2.VideoCapture(str(VIDEO_IN))
f_idx = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    for d in frames_track[f_idx]:
        if d["cls"] != 1: continue
        c = color_torso(frame, d["bbox"])
        if c is not None:
            col_por_id[d["id"]].append(c)
    f_idx += 1
cap.release()

ids_validos = [i for i, cs in col_por_id.items() if len(cs) >= 3]
col_id = np.array([np.median(col_por_id[i], axis=0) for i in ids_validos])

crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-4)
_, labels_k, centers = cv2.kmeans(col_id.astype(np.float32), 2, None, crit, 20, cv2.KMEANS_PP_CENTERS)
labels_k = labels_k.ravel()
equipo_de_id = {ids_validos[j]: int(labels_k[j]) for j in range(len(ids_validos))}

def realzar(c):
    c = np.array(c, dtype=float)
    return tuple(int(v) for v in (c / (c.max() + 1e-6) * 255))
color_equipo_bgr = {k: realzar(centers[k]) for k in (0, 1)}

equipo_det, id_final_det = {}, {}
for f_idx, dets in enumerate(frames_track):
    for d in dets:
        if d["cls"] not in (1, 2): continue
        id_final_det[(f_idx, d["id"])] = d["id"]
        equipo_det[(f_idx, d["id"])] = equipo_de_id.get(d["id"], -1)

n0 = sum(1 for v in equipo_de_id.values() if v == 0)
n1 = sum(1 for v in equipo_de_id.values() if v == 1)
print(f"Identidades clasificadas: {len(ids_validos)}")
print(f"Equipo A(0)={color_equipo_bgr[0]}: {n0} | Equipo B(1)={color_equipo_bgr[1]}: {n1}")

Identidades clasificadas: 33
Equipo A(0)=(121, 241, 254): 19 | Equipo B(1)=(253, 254, 247): 14


In [16]:
# ---- Roles (árbitro/portero/jugador) + corrección del equipo del portero ----
PORTERIAS = np.array([[0.0, 34.0], [105.0, 34.0]])
X_AREA_PORTERO, RADIO_GK_MAX = 8.0, 14.0
nombres = {0: "A", 1: "B", -1: "-"}

campo_id = defaultdict(list)
for f_idx, dets in enumerate(frames_track):
    for d in dets:
        if d["cls"] not in (1, 2): continue
        p = campo_det.get((f_idx, d["id"]))
        if p is not None:
            campo_id[d["id"]].append(p)

stats = {}
for nid, pts in campo_id.items():
    a = np.array(pts)
    if len(a) >= 5:
        med = np.median(a, axis=0)
        stats[nid] = {"med": med, "radio": np.percentile(np.linalg.norm(a-med, axis=1), 90),
                      "x": med[0]}

cls_por_nid = defaultdict(lambda: {1: 0, 2: 0})
for fr in frames_track:
    for d in fr:
        if d["cls"] in (1, 2):
            cls_por_nid[d["id"]][d["cls"]] += 1

def dentro(p): return -5 <= p[0] <= 110 and -5 <= p[1] <= 73

rol_de, equipo_seg = {}, {}
for nid in sorted({d["id"] for fr in frames_track for d in fr if d["cls"] in (1,2)}):
    eq = equipo_de_id.get(nid, -1); equipo_seg[nid] = eq
    c = cls_por_nid[nid]; total_cls = c[1] + c[2]; st = stats.get(nid)
    if total_cls > 0 and c[2]/total_cls >= 0.5:
        rol_de[nid] = "arbitro"; equipo_seg[nid] = -1; continue
    if st is None:
        rol_de[nid] = "jugador" if eq in (0,1) else "fuera"; continue
    if not dentro(st["med"]):
        rol_de[nid] = "fuera"; continue
    pegado = st["x"] < X_AREA_PORTERO or st["x"] > (105 - X_AREA_PORTERO)
    if pegado and st["radio"] <= RADIO_GK_MAX:
        rol_de[nid] = "portero"; continue
    rol_de[nid] = "jugador" if eq in (0,1) else "fuera"

# Corrección del equipo del portero por cercanía
for nid in list(rol_de.keys()):
    if rol_de.get(nid) != "portero": continue
    gk_pos = {}
    for f_idx, dets in enumerate(frames_track):
        for d in dets:
            if d["id"] == nid:
                p = campo_det.get((f_idx, d["id"]))
                if p: gk_pos[f_idx] = p
    votos = {0: 0.0, 1: 0.0}
    for f_idx, gp in gk_pos.items():
        cercanos = []
        for d in frames_track[f_idx]:
            if d["cls"] != 1 or rol_de.get(d["id"]) != "jugador": continue
            eqo = equipo_seg.get(d["id"]); p = campo_det.get((f_idx, d["id"]))
            if eqo in (0,1) and p: cercanos.append((np.hypot(p[0]-gp[0], p[1]-gp[1]), eqo))
        cercanos.sort()
        for dist, eqo in cercanos[:3]: votos[eqo] += 1.0/(dist+1.0)
    if votos[0]+votos[1] > 0:
        eqc = 0 if votos[0] > votos[1] else 1
        if equipo_seg.get(nid) != eqc:
            print(f"PORTERO {nid}: corregido equipo {nombres.get(equipo_seg.get(nid),'-')} -> {nombres[eqc]}")
            equipo_seg[nid] = eqc
            for (f,i),n in id_final_det.items():
                if n == nid: equipo_det[(f,i)] = eqc

n_roles = Counter(rol_de.values())
print(f"Roles: {dict(n_roles)}")

PORTERO 87: corregido equipo A -> B
PORTERO 96: corregido equipo B -> A
Roles: {'jugador': 30, 'fuera': 1, 'arbitro': 16, 'portero': 2}


In [18]:
# ---- Exportar el JSON integrado ----
def f(v):      # float nativo (json no serializa float32 de numpy)
    return float(v)
def lista(seq):
    return [float(x) for x in seq] if seq is not None else None

salida = {"video": VIDEO_FILE, "n_frames": int(N_FRAMES),
          "color_equipo_bgr": {str(k): [int(x) for x in v] for k, v in color_equipo_bgr.items()},
          "frames": []}

for f_idx in range(N_FRAMES):
    dets_frame = []
    for d in frames_track[f_idx]:
        nid = int(d["id"])
        if d["cls"] == 0:   # balón
            campo = campo_det.get((f_idx, d["id"]))
            dets_frame.append({"id": nid, "rol": "balon", "equipo": -1,
                               "bbox": lista(d["bbox"]), "campo": lista(campo)})
        else:
            rol = rol_de.get(d["id"], "fuera")
            if rol == "fuera": continue
            campo = campo_det.get((f_idx, d["id"]))
            dets_frame.append({"id": nid, "rol": rol, "equipo": int(equipo_seg.get(d["id"], -1)),
                               "bbox": lista(d["bbox"]), "campo": lista(campo)})
    salida["frames"].append(dets_frame)

with open(OUT_JSON, "w", encoding="utf-8") as fh:
    json.dump(salida, fh)

roles_tot = Counter(d["rol"] for fr in salida["frames"] for d in fr)
print(f"Guardado: {OUT_JSON.name}")
print(f"Roles totales: {dict(roles_tot)}")
print(f"Colores: A={color_equipo_bgr[0]}, B={color_equipo_bgr[1]}")

Guardado: integrado_test_27.json
Roles totales: {'jugador': 15825, 'balon': 504, 'arbitro': 224, 'portero': 174}
Colores: A=(121, 241, 254), B=(253, 254, 247)


In [20]:
# ---- Post-proceso test_27: portero duplicado + ELIMINAR fragmentos (deja JSON limpio) ----
import json, numpy as np
from collections import defaultdict, Counter

with open(OUT_JSON, encoding="utf-8") as fh:
    data = json.load(fh)
frames = data["frames"]
def en_campo(c): return c is not None and -5<=c[0]<=110 and -5<=c[1]<=73

# (1) Portero duplicado: el menos estable de cada portería pasa a jugador
gk_info = defaultdict(lambda: {"n":0, "pos":[]})
for fr in frames:
    for d in fr:
        if d.get("rol")=="portero" and en_campo(d.get("campo")):
            gk_info[d["id"]]["n"]+=1; gk_info[d["id"]]["pos"].append(d["campo"])
por_lado = defaultdict(list)
for gid, inf in gk_info.items():
    p = np.median(inf["pos"], axis=0); lado = "izq" if p[0]<52.5 else "der"
    por_lado[lado].append((inf["n"], gid))
falsos = set()
for lado, lista in por_lado.items():
    lista.sort(reverse=True)
    for n, gid in lista[1:]:
        falsos.add(gid); print(f"Portero falso ID {gid} ({n} det) -> jugador")
for fr in frames:
    for d in fr:
        if d.get("id") in falsos and d.get("rol")=="portero": d["rol"]="jugador"

# (2) Quedarse con 11 por equipo, ELIMINAR el resto
det_id = defaultdict(int)
for fr in frames:
    for d in fr:
        if d.get("rol") in ("jugador","portero") and d.get("equipo") in (0,1) and en_campo(d.get("campo")):
            det_id[(d["equipo"], d["id"])]+=1
titulares = set()
for eq in (0,1):
    # portero del equipo primero
    for g in gk_info:
        if g in falsos: continue
        eqs=[d.get("equipo") for fr in frames for d in fr if d.get("id")==g]
        if eqs and eqs[0]==eq and (eq,g) in det_id: titulares.add((eq,g))
    for n,i in sorted([(n,i) for (e,i),n in det_id.items() if e==eq], reverse=True):
        if len([1 for (ee,_) in titulares if ee==eq])>=11: break
        titulares.add((eq,i))

# eliminar detecciones de jugadores no titulares
for fr in frames:
    fr[:] = [d for d in fr if not (d.get("rol") in ("jugador","portero")
             and d.get("equipo") in (0,1) and (d["equipo"], d["id"]) not in titulares)]

with open(OUT_JSON, "w", encoding="utf-8") as fh:
    json.dump(data, fh)
roles = Counter(d["rol"] for fr in frames for d in fr)
ids_eq = defaultdict(set)
for fr in frames:
    for d in fr:
        if d.get("rol") in ("jugador","portero") and d.get("equipo") in (0,1):
            ids_eq[d["equipo"]].add(d["id"])
print(f"JSON limpio guardado. Roles: {dict(roles)}")
print(f"Titulares: A={len(ids_eq[0])}, B={len(ids_eq[1])}")

JSON limpio guardado. Roles: {'jugador': 14263, 'balon': 504, 'arbitro': 224, 'portero': 169}
Titulares: A=11, B=11


In [21]:
# ---- Setup motor táctico (versión que respeta es_fragmento, sirve para cualquier vídeo) ----
import json, numpy as np
from pathlib import Path
from collections import defaultdict
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Arc

REPO_ROOT = Path("../..").resolve()
NOMBRE = "test_27"          # <--- cambia esto según el vídeo (test_27 o A1606b0e6_0_62)
JSON_IN = REPO_ROOT / "data" / "tracking" / f"integrado_{NOMBRE}.json"
OUT_DIR = REPO_ROOT / "outputs" / "tactico"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(JSON_IN, encoding="utf-8") as fh:
    data = json.load(fh)
frames = data["frames"]
N_FRAMES = data["n_frames"]
color_equipo_bgr = {int(k): tuple(v) for k, v in data["color_equipo_bgr"].items()}
col_eq = {eq: tuple(np.array(bgr[::-1]) / 255.0) for eq, bgr in color_equipo_bgr.items()}

MARGEN = 5.0
def en_campo(c):
    return c is not None and -MARGEN <= c[0] <= 105 + MARGEN and -MARGEN <= c[1] <= 68 + MARGEN

def es_valido(d):
    """Jugador/portero de un equipo, con posición, que NO sea fragmento marcado."""
    return (d.get("rol") in ("jugador", "portero") and d.get("equipo") in (0, 1)
            and en_campo(d.get("campo")) and not d.get("es_fragmento"))

def jugadores_frame(f_idx, equipo, incluir_portero=False):
    roles = ("jugador", "portero") if incluir_portero else ("jugador",)
    return [d["campo"] for d in frames[f_idx]
            if d.get("rol") in roles and d.get("equipo") == equipo
            and en_campo(d.get("campo")) and not d.get("es_fragmento")]

print(f"JSON: {JSON_IN.name} | Frames: {N_FRAMES}")
print(f"Equipos: A={color_equipo_bgr[0]}, B={color_equipo_bgr[1]}")
# verificar que el filtro deja 11 por equipo
from collections import Counter
ids_val = defaultdict(set)
for fr in frames:
    for d in fr:
        if es_valido(d): ids_val[d["equipo"]].add(d["id"])
print(f"Jugadores válidos (sin fragmentos): A={len(ids_val[0])}, B={len(ids_val[1])}")

JSON: integrado_test_27.json | Frames: 750
Equipos: A=(121, 241, 254), B=(253, 254, 247)
Jugadores válidos (sin fragmentos): A=11, B=11
